# Prediction uncertainty

Parameter confidence intervals ([identifiability](parameter_identifiability.ipynb)) answer how well the data determines the parameters.
Often the more relevant question is how well it determines a *prediction*: the value of a variable, flux or derived quantity at some time, possibly in a condition that wasn't measured.
A prediction can be well determined even if the parameters aren't, and the other way around.

`mxlpy.identify` computes likelihood-based answers to that question:

- `prediction_profile`: the profile likelihood and confidence interval of one prediction
- `prediction_band`: pointwise confidence bands of a prediction over time
- `validation_profile`: whether a new data point is consistent with the calibrated model
- `profile_likelihood_2d`: joint confidence regions of two parameters, or of a parameter and a prediction

Unlike bands from parameter ensembles or Monte Carlo samples, these don't depend on how samples were drawn and have the right coverage even for sloppy or non-identifiable models.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mxlpy import Model, Simulator, fit, fns, identify, plot

We use the SIR model of the identifiability tutorial, but only observe the first 25 days of the epidemic.

In [ ]:
def sir() -> Model:
    return (
        Model()
        .add_variables({"s": 0.9, "i": 0.1, "r": 0.0})
        .add_parameters({"beta": 0.2, "gamma": 0.1})
        .add_reaction(
            "infection",
            fns.mass_action_2s,
            args=["s", "i", "beta"],
            stoichiometry={"s": -1, "i": 1},
        )
        .add_reaction(
            "recovery",
            fns.mass_action_1s,
            args=["i", "gamma"],
            stoichiometry={"i": -1, "r": 1},
        )
    )


times = np.linspace(0, 25, 26)
truth = Simulator(sir()).simulate_time_course(times).get_result().unwrap_or_err()
rng = np.random.default_rng(42)
data = truth.variables.loc[1:, ["i", "r"]]
data = data + rng.normal(0, 0.01, data.shape)
nll = fit.GaussianNll(sigma=0.01)

res = fit.time_course(
    sir(),
    p0={"beta": 0.3, "gamma": 0.05},
    data=data,
    loss_fn=nll,
    minimizer=fit.LeastSquaresMinimizer(),
).unwrap_or_err()
res.best_pars

## Prediction profiles

A `Prediction` names the predicted output, its time (`None` for the steady state) and optionally a condition: `parameters` and `y0` overriding the fitted model's, or a `protocol`.

The prediction profile is the best objective of all parameters predicting a value `z`.
Thresholding it at `chi2.ppf(alpha, 1) / 2` (with a likelihood loss like `fit.GaussianNll`) gives the confidence interval of the prediction.
Here we ask for the number of infected at day 60, long after the data ends.

In [ ]:
prediction = identify.Prediction("i", time=60.0)
pred = identify.prediction_profile(
    sir(), data, res, prediction=prediction, loss_fn=nll, transform="log"
)
_ = plot.prediction_profile(pred)
pred.ci

The profile is traced by refits with a pseudo data point `z ± σ` of the prediction.
The best fit with it is also the best fit among all parameters predicting the same value, so each refit gives an exact point of the profile, whatever `σ`.
`pred.parameters` holds the refitted parameters along it, `pred.z` the pseudo data points.

A prediction in another condition, here a treatment doubling the recovery rate after the observed period, only needs the overriding values:

In [ ]:
treated = identify.prediction_profile(
    sir(),
    data,
    res,
    prediction=identify.Prediction("i", time=60.0, parameters={"gamma": 0.2}),
    loss_fn=nll,
    transform="log",
)
treated.value, treated.ci

## Prediction bands

`prediction_band` computes the confidence interval at many times.
The default `method="integrated"` tracks the bounds along time: the refit at the bound of one time predicts the next one, and a few corrective refits move it onto the threshold, which is much cheaper for dense time grids.
`method="pointwise"` computes a full prediction profile at every time instead (in parallel) and keeps them in `band.profiles`.

In [ ]:
band = identify.prediction_band(
    sir(),
    data,
    res,
    prediction="i",
    times=np.linspace(0, 100, 51),
    loss_fn=nll,
    transform="log",
)
fig, ax = plot.prediction_band(band)
ax.scatter(data.index, data["i"], color="0.3", s=8, label="data")
ax.legend()
plt.show()

## Predictions of non-identifiable models

If the infection rate is the product of a contact rate `c` and a transmission probability `beta`, the data can't tell them apart: their profiles are flat.
The predictions only depend on the product, so they are just as well determined as before.

In [ ]:
def sir_contacts() -> Model:
    return (
        Model()
        .add_variables({"s": 0.9, "i": 0.1, "r": 0.0})
        .add_parameters({"beta": 0.2, "gamma": 0.1, "c": 1.0})
        .add_derived("rate", fns.mul, args=["beta", "c"])
        .add_reaction(
            "infection",
            fns.mass_action_2s,
            args=["s", "i", "rate"],
            stoichiometry={"s": -1, "i": 1},
        )
        .add_reaction(
            "recovery",
            fns.mass_action_1s,
            args=["i", "gamma"],
            stoichiometry={"i": -1, "r": 1},
        )
    )


res_c = fit.time_course(
    sir_contacts(),
    p0={"beta": 0.3, "gamma": 0.05, "c": 1.5},
    data=data,
    loss_fn=nll,
    minimizer=fit.LeastSquaresMinimizer(),
).unwrap_or_err()
bounds = {"beta": (1e-3, 10.0), "c": (1e-2, 100.0)}
pred_c = identify.prediction_profile(
    sir_contacts(),
    data,
    res_c,
    prediction=prediction,
    loss_fn=nll,
    transform="log",
    bounds=bounds,
)
pred.ci, pred_c.ci

## Validation profiles

Before (or after) measuring a new data point, `validation_profile` tells which values are consistent with the calibrated model.
It adds the data point with its measurement error `sigma` to the fit and steps its value: a measured value is consistent if adding it increases the objective by less than the threshold (a likelihood-ratio test).
The consistent values are wider than the prediction's confidence interval, as they include the measurement error.

In [ ]:
val = identify.validation_profile(
    sir(), data, res, prediction=prediction, sigma=0.01, loss_fn=nll, transform="log"
)
_ = plot.prediction_profile(val)
print(val.validation_ci)
measured = float(
    Simulator(sir()).simulate(60).get_result().unwrap_or_err().variables["i"].iloc[-1]
)
val.is_consistent(measured), val.is_consistent(measured + 0.1)

## Two-dimensional profiles

1D profiles show *that* a parameter is uncertain, not which others compensate for it.
`profile_likelihood_2d` profiles two parameters jointly, refitting all others at each point, and gives their joint confidence region (threshold `chi2.ppf(alpha, 2) / 2`).
With only early data, `beta` and `gamma` compensate along the initial growth rate `0.9 beta - gamma`, so the region is a narrow tilted valley.

- `method="slices"` (default) profiles `x`, then at each of its points profiles `y` with `x` fixed until the threshold, so no fits are spent outside the region. `y` can also be a `Prediction`
- `method="grid"` refits on an `n_steps` x `n_steps` grid with warm starts along each row

In [ ]:
pl2d = identify.profile_likelihood_2d(
    sir(), data.loc[:10], res, x="beta", y="gamma", loss_fn=nll
)
_ = plot.profile_likelihood_2d(pl2d)
pl2d.region().head()

`region()` gives the boundary of the region as a `y` interval per `x` slice, `pl2d.points` all refitted points with their increase of the objective and `pl2d.parameters` the refitted parameters.
Points much worse than a neighbour (failed fits) are refitted from the neighbour's parameters, `pl2d.n_repaired` counts them.